# Unitree G1: GPU RL pipeline (MJX + custom PPO)

Settings: **Accelerator = GPU T4 x2**, **Internet = On**, Persistence = Files.

Run top to bottom. Phase 0 = environment setup + throughput benchmark + training-time estimate.

## 1. Get the code
Either upload `g1_rl.zip` as a Kaggle Dataset (Add Data → it appears under `/kaggle/input`), or set `GIT_URL` to your repo.

In [ ]:
import os, shutil, glob, subprocess
GIT_URL = ""   # e.g. "https://github.com/<you>/g1_rl.git"; leave empty to use an attached dataset
WORK = "/kaggle/working/g1_rl"
if not os.path.exists(os.path.join(WORK, "pyproject.toml")):
    if GIT_URL:
        subprocess.run(["git", "clone", "--depth", "1", GIT_URL, WORK], check=True)
    else:
        hits = [p for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True) if "g1_rl" in open(p).read()]
        assert hits, "Attach the g1_rl dataset or set GIT_URL"
        shutil.copytree(os.path.dirname(hits[0]), WORK)
os.chdir(WORK)
print(os.getcwd(), sorted(os.listdir()))

## 2. Install GPU JAX + MuJoCo/MJX

In [ ]:
!bash scripts/install_kaggle.sh
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 3. Setup check: devices, model, and MJX (GPU, float32) vs MuJoCo (CPU, float64) agreement

In [ ]:
!python scripts/check_setup.py --platform gpu

## 4. Throughput benchmark + training-time estimate
`--scaling` also measures 1 GPU so you can see the 2×T4 speed-up. Takes ~10–20 min (mostly compilation, once per batch size).

In [ ]:
!python scripts/benchmark.py --platform gpu --scaling --envs 1024,2048,4096,8192 --out outputs/benchmark.json

Optional: compare the explicit per-device mode (may be faster since devices never sync; it crashed on CPU multi-device in testing, so it is not the default).

In [ ]:
# !python scripts/benchmark.py --platform gpu --parallel shard_map --envs 4096,8192 --out outputs/benchmark_shardmap.json

In [ ]:
import json
r = json.load(open("outputs/benchmark.json"))
print(r["devices"], "| best:", r["best"]["n_envs"], "envs ->", f'{r["best"]["env_steps_per_s"]:,.0f} env-steps/s')
for e in r["estimate"]:
    print(f'{e["env_steps"]/1e6:4.0f}M steps -> {e["hours"]:.1f} h  (~{e["kaggle_sessions_12h"]} session(s))')

## 5. Unit tests (optional)

In [ ]:
!pip install -q pytest && G1RL_TEST_PLATFORM=gpu python -m pytest -q

## Next phases
Environment, rewards, domain randomization, PPO training, and evaluation cells are added in later phases.